# 06c — Trayectorias de distancia a la línea de pobreza, era histórica (2003T3-2015T2)

Extiende `data/05_trayectoria_distancia/{pares_h1,pares_h4}/` a la era `"historica"` con el
mismo cálculo que `06_trayectorias_distancia.ipynb` (`src/trayectoria_distancia.py`:
variaciones, descomposición, componente común/propio, tendencia previa, dirección y
transición), mismas particiones (`ANO4=`/`TRIMESTRE=` de origen) y mismas columnas, más:

- `era` (= `era_t` = `era_th` = `"historica"`).
- **Peso**: no existe `PONDIH` antes de 2016. `PONDIH_t/th` quedan en `NA` (mismo esquema que la
  era regular) y se agregan `peso_hogar_t/th` (`PONDERA` del hogar, ver
  `ponderadores.peso_hogar_historico`) y `n_miembros_t/th`. El componente común usa
  `peso_hogar_t`. Para juntar eras: `cortes_trayectoria.peso_hogar_era` +
  `peso_normalizado_trimestre` (en `06d`).
- **Imputación de ingresos** (decisión tomada sobre `05d_no_respuesta_historica.ipynb`):
  `imputado_t/th` (`IDIMPH != 00000`), `fraccion_min_t/th`, `fraccion_max_t/th`
  (`src/imputacion_ingresos.py`), y `par_valido_c = par_valido & ~imputado_t & ~imputado_th`
  (`par_valido_log_c` igual sobre `par_valido_log`). **Los ratios no se anulan**: la variante A
  (imputados adentro) sigue disponible con `par_valido`.
- **Flags de corte** (`src/cortes_trayectoria.py`, marcan, no excluyen): período intervenido
  INDEC 2007-2015, ruptura de `CODUSU` 2013-2014 (par y tendencia previa), hueco 2007T3 (par y
  tendencia previa), trimestres con `ITF==0` declarado elevado, y 2020T2 (nunca en esta era).

**Nota**: `variacion_comun_*` usa información de `t+h` — es descriptivo, nunca una feature.

**Lee** `data/02b_distancia_linea_pobreza/hogar/` (era `"historica"`), `data/03_panel/` y
`data/01_armonizado/` (`IDIMPH`/`IDIMPP` y montos, no están en `02_nucleo`). **Escribe** solo
particiones de origen `<= 2015T2` en `data/05_trayectoria_distancia/` — aditivo, `06` nunca las
pisa.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import cortes_trayectoria as ct
import imputacion_ingresos as ii
import panel
import pipeline_meta as pm
import trayectoria_distancia
from armonizacion import trimestres_publicados

DISTANCIA_LINEA_POBREZA = REPO / "data" / "02b_distancia_linea_pobreza" / "hogar"
ARMONIZADO = REPO / "data" / "01_armonizado"
PANEL_DIR = REPO / "data" / "03_panel"
TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
META = REPO / "data" / "meta"
COMMIT_GIT = pm.obtener_commit_git(REPO)
ERA = "historica"

TRIMESTRES_HIST = [t for t in trimestres_publicados() if t <= (2015, 2)]
print(f"{len(TRIMESTRES_HIST)} trimestres históricos")

## Distancia (Capa 1) y trimestres con `ITF==0` elevado

Se carga la distancia histórica una vez por trimestre. El conjunto de trimestres con % de
hogares con `ITF==0` por encima del umbral se calcula sobre **todas** las particiones de `02b`
(ambas eras), excluyendo solo la no respuesta confirmada (`ingreso_no_declarado==True`, era
regular) — así es comparable entre eras — y se registra en `corridas.csv` (el umbral de 2% está cerca de la frontera de los
datos actuales).

In [ ]:
distancia_por_trimestre = {}
pct_itf0 = {}
for p in sorted(DISTANCIA_LINEA_POBREZA.glob("ANO4=*/TRIMESTRE=*/data.parquet")):
    anio, trimestre = int(p.parent.parent.name.split("=")[1]), int(p.parent.name.split("=")[1])
    df = pd.read_parquet(p)
    pct_itf0[(anio, trimestre)] = ct.pct_itf_cero_declarado(df)
    if (anio, trimestre) in TRIMESTRES_HIST:
        assert (df["era"] == ERA).all(), f"{anio}T{trimestre}: partición no histórica en el rango histórico"
        assert "PONDIH" not in df
        df["PONDIH"] = pd.array([pd.NA] * len(df), dtype="Int32")
        distancia_por_trimestre[(anio, trimestre)] = df

TRIMESTRES_ITF0 = ct.trimestres_itf0_elevado(pct_itf0, ct.UMBRAL_ITF0_PCT)
print(f"{len(distancia_por_trimestre)} particiones históricas cargadas")
print(f"umbral {ct.UMBRAL_ITF0_PCT}% -> trimestres con ITF==0 declarado elevado: {TRIMESTRES_ITF0}")
display(pd.Series(pct_itf0, name="pct_itf_cero_declarado").sort_values(ascending=False).head(8).round(2))

## Imputación por hogar

`imputacion_ingresos.fraccion_itf_imputada` sobre `01_armonizado`, una vez por trimestre
(mismo cálculo que `05d`).

In [ ]:
t0 = time.time()
imputacion_por_trimestre = {}
for anio, trimestre in TRIMESTRES_HIST:
    hogar = ii.leer_marcas_imputacion(anio, trimestre, ARMONIZADO, "hogar")
    individual = ii.leer_marcas_imputacion(anio, trimestre, ARMONIZADO, "individual")
    fraccion = ii.fraccion_itf_imputada(hogar, individual)
    imputacion_por_trimestre[(anio, trimestre)] = fraccion[
        ["CODUSU", "NRO_HOGAR", "alguna_imputacion", "fraccion_min", "fraccion_max"]].rename(
        columns={"alguna_imputacion": "imputado"})
print(f"imputación calculada para {len(imputacion_por_trimestre)} trimestres, {time.time() - t0:.0f}s")

## Helper de enriquecimiento por par de origen

Mismo cálculo que `06` (`construir_pares_distancia` + componente común/propio + tendencia
previa), con `peso_hogar_t` como peso del componente común, y además las columnas de peso por
punta, imputación, `par_valido_c`/`par_valido_log_c` y flags de corte.

In [ ]:
def _agregar_lado(pares: pd.DataFrame, extra: pd.DataFrame, sufijo: str) -> pd.DataFrame:
    renombrado = extra.rename(columns={c: f"{c}_{sufijo}" for c in extra.columns if c != "CODUSU"})
    return pares.merge(renombrado, on=["CODUSU", f"NRO_HOGAR_{sufijo}"], how="left")


def construir_pares_enriquecidos(origen: tuple[int, int], destino: tuple[int, int], pares_hogar: pd.DataFrame,
                                 previo_1: pd.DataFrame | None, previo_4: pd.DataFrame | None) -> pd.DataFrame:
    distancia_t, distancia_th = distancia_por_trimestre[origen], distancia_por_trimestre[destino]
    pares = trayectoria_distancia.construir_pares_distancia(distancia_t, distancia_th, pares_hogar)
    for sufijo, distancia, q in (("t", distancia_t, origen), ("th", distancia_th, destino)):
        pares = _agregar_lado(pares, distancia[["CODUSU", "NRO_HOGAR", "peso_hogar", "n_miembros"]], sufijo)
        pares = _agregar_lado(pares, imputacion_por_trimestre[q], sufijo)
    for columna_variacion, sufijo in (("variacion_simetrica", "simetrica"), ("variacion_log", "log")):
        pares = trayectoria_distancia.agregar_componente_comun_y_propio(
            pares, columna_variacion, "peso_hogar_t", ["ANO4_t", "TRIMESTRE_t"], f"trimestre_{sufijo}")
        pares = trayectoria_distancia.agregar_componente_comun_y_propio(
            pares, columna_variacion, "peso_hogar_t", ["ANO4_t", "TRIMESTRE_t", "AGLOMERADO_t"], f"aglomerado_{sufijo}")
    pares = trayectoria_distancia.agregar_tendencia_previa(pares, previo_1, previo_4)
    pares["era"] = ERA
    sin_imputar = (pares["imputado_t"] == False).fillna(False) & (pares["imputado_th"] == False).fillna(False)  # noqa: E712
    pares["par_valido_c"] = pares["par_valido"] & sin_imputar
    pares["par_valido_log_c"] = pares["par_valido_log"] & sin_imputar
    return ct.marcar_cortes_par(pares, TRIMESTRES_ITF0)

## `pares_h1` y `pares_h4`

Orígenes en orden cronológico; `h=1` primero (su propio `pares_h1` de `t-1`/`t-4` alimenta la
tendencia previa), `h=4` después contra el `pares_h1` ya completo — igual que `06`. Solo
orígenes con `panel.tiene_destino_valido` (ningún destino cruza a 2016T2+).

In [ ]:
pares_por_h: dict[int, dict[tuple[int, int], pd.DataFrame]] = {1: {}, 4: {}}
origenes_sin_pares: list[dict] = []
duraciones = {}
for h in (1, 4):
    t0 = time.time()
    for anio, trimestre in TRIMESTRES_HIST:
        if not panel.tiene_destino_valido(anio, trimestre, h):
            continue
        destino = panel.sumar_trimestres(anio, trimestre, h)
        p_panel = pm.ruta_particion(PANEL_DIR, f"pares_h{h}", anio, trimestre)
        if destino not in distancia_por_trimestre or not p_panel.exists():
            continue
        previo_1 = pares_por_h[1].get(panel.sumar_trimestres(anio, trimestre, -1))
        previo_4 = pares_por_h[1].get(panel.sumar_trimestres(anio, trimestre, -4))
        pares_hogar = pd.read_parquet(p_panel)
        if pares_hogar.empty:
            origenes_sin_pares.append({"h": h, "origen": f"{anio}T{trimestre}", "destino": f"{destino[0]}T{destino[1]}"})
            continue
        pares = construir_pares_enriquecidos((anio, trimestre), destino, pares_hogar, previo_1, previo_4)
        assert (pares["era_t"] == ERA).all() and (pares["era_th"] == ERA).all()
        pares_por_h[h][(anio, trimestre)] = pares
        salida = pm.ruta_particion(TRAYECTORIA, f"pares_h{h}", anio, trimestre)
        salida.parent.mkdir(parents=True, exist_ok=True)
        pares.to_parquet(salida, index=False)
    duraciones[h] = time.time() - t0
    n = sum(len(df) for df in pares_por_h[h].values())
    print(f"pares_h{h}: {len(pares_por_h[h])} orígenes, {n:,} pares, {duraciones[h]:.0f}s")
print("orígenes con 0 pares en 03_panel (sin partición escrita):", origenes_sin_pares)

**Anomalía documentada**: un origen con 0 pares en `03_panel` no se escribe (no hay nada que
escribir) y queda listado arriba. Con los datos actuales es `2013T2` con `h=4`, la ruptura de
`CODUSU` 2013-2014 ya diagnosticada en `04_trayectorias` (la cohorte de entrada 2013T1/T2 no
reaparece en 2014): no es una falla del cálculo.

## Frontera 2015T2 / 2016T2

Ningún par histórico puede llegar a 2016T2 o después: `CODUSU` cambia de formato entre eras (0%
de vinculación, ver `decisiones.md`) y `panel.tiene_destino_valido` corta esos destinos. Se
verifica sobre lo escrito, no se asume.

In [ ]:
for h in (1, 4):
    todos = pd.concat(pares_por_h[h].values(), ignore_index=True)
    destino_max = todos[["ANO4_th", "TRIMESTRE_th"]].apply(tuple, axis=1).max()
    assert destino_max <= (2015, 4), f"h={h}: destino {destino_max} cruza la frontera"
    assert (todos["era"] == ERA).all()
    print(f"h={h}: destino máximo {destino_max[0]}T{destino_max[1]} -- ningún par cruza a 2016T2+")

## Registro de la corrida

In [ ]:
n_pares = sum(len(df) for d in pares_por_h.values() for df in d.values())
pm.registrar_corrida(
    META / "corridas.csv", "06c_trayectorias_distancia_historica",
    {"era": ERA, "criterio_imputacion": "C (par_valido_c marca, ratios sin anular)",
     "umbral_itf0_pct": ct.UMBRAL_ITF0_PCT, "trimestres_itf0_elevado": TRIMESTRES_ITF0,
     "origenes_sin_pares": origenes_sin_pares},
    len(pares_por_h[1]) + len(pares_por_h[4]), n_pares, sum(duraciones.values()), COMMIT_GIT)
print(f"corrida registrada: {n_pares:,} pares")

## Reporte: validez, imputación y flags de corte, por horizonte

In [ ]:
filas = []
for h in (1, 4):
    todos = pd.concat(pares_por_h[h].values(), ignore_index=True)
    fila = {"h": h, "n_pares": len(todos), "n_par_valido": int(todos["par_valido"].sum()),
            "n_par_valido_c": int(todos["par_valido_c"].sum()), "n_par_valido_log": int(todos["par_valido_log"].sum()),
            "n_par_valido_log_c": int(todos["par_valido_log_c"].sum()),
            "pct_alguna_punta_imputada": 100 * ((todos["imputado_t"] == True) | (todos["imputado_th"] == True)).mean(),  # noqa: E712
            "pct_con_tendencia_previa": 100 * todos["variacion_simetrica_previa"].notna().mean()}
    for flag in ct.FLAGS_CORTE:
        fila[f"n_{flag}"] = int(todos[flag].sum())
    filas.append(fila)
reporte = pd.DataFrame(filas)
reporte.to_csv(META / "trayectoria_distancia_historica_reporte.csv", index=False)
display(reporte.T)